# CoMind GCI-001: Google access setup without a terminal

Use a standard **free CPU runtime**. No GPU, Colab Enterprise, paid plan or AI assistant is needed.
Open in the Safari app if Google's sign-in rejects an embedded browser.

1. Run the **Show setup buttons** cell below using its triangle button.
2. Tap **Sign in and check** and choose the Google account that administers CoMind.
3. After the check succeeds, tap **Apply reviewed setup**, then **Verify setup**.
4. Download the safe receipt and return it to the existing GCI-001 Work conversation.

Running all cells only displays buttons. It does not sign in or change IAM.
The Apply button can enable the five required APIs, create the dedicated pool/provider if absent,
and grant only the restricted GitHub identity permission to use the existing service account.
It stops on unexpected existing trust, project roles or user-managed keys.
The account you sign in with retains its own permissions; this notebook's code restricts its
operations to the reviewed Test setup. It does not create a narrower human OAuth identity.

Project: `comind-513807` (`373198958202`). Lane: issue [#544](https://github.com/jlhogarth/CoMind/issues/544),
draft PR [#545](https://github.com/jlhogarth/CoMind/pull/545).
No service-account keys, Google Drive mounting, model calls, GitHub credentials or Production deployment.
Notebook source embeds the reviewed script and verifies its SHA-256 before each invocation.
The receipt contains statuses and nonsecret configuration only. Disconnect and delete the
runtime after downloading the receipt to end this temporary admin session.

**Verification boundary:** local tests cover controls and bootstrap behavior. Your Google login,
actual IAM changes and iPhone interaction still require live verification. If sign-in fails,
stop here rather than copying auth codes or repeating the old terminal instructions.


In [ ]:
#@title Show setup buttons
"""Button-driven Colab adapter for the existing GCI IAM bootstrap.

No credentials, raw provider errors, or project policy are saved in receipts.
"""
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
from datetime import datetime, timezone
from pathlib import Path

PROJECT = "comind-513807"


class SetupPrerequisiteError(RuntimeError):
    """Safe, locally generated diagnostic, containing no authentication data."""


def classify_failure(stderr, returncode):
    message = stderr.lower()
    if "active account" in message or "reauth" in message or "credential" in message:
        return "GOOGLE_SIGN_IN_REQUIRED"
    if "permission" in message or "forbidden" in message:
        return "GOOGLE_PERMISSION_DENIED"
    if returncode == 3:
        return "EXISTING_CONFIGURATION_REQUIRES_REVIEW"
    if returncode == 4:
        return "RUNTIME_DEPENDENCY_MISSING"
    if returncode == 5:
        return "SETUP_INCOMPLETE"
    return "GOOGLE_COMMAND_FAILED"


def run_setup(source, expected_sha256, mode, runner=subprocess.run):
    if mode not in ("--audit", "--apply", "--verify"):
        raise ValueError("Unsupported setup mode")
    if hashlib.sha256(source.encode()).hexdigest() != expected_sha256:
        raise ValueError("Reviewed setup bytes do not match their digest")
    receipt = {
        "schema_version": 1, "lane": "GCI-001", "project_id": PROJECT,
        "project_number": "373198958202", "mode": mode,
        "script_sha256": expected_sha256,
        "started_utc": datetime.now(timezone.utc).isoformat(),
        "ready": False, "gemini_requests": 0, "checks": [],
    }
    with tempfile.TemporaryDirectory(prefix="gci-setup-") as directory:
        path = Path(directory) / "bootstrap-google-oidc.sh"
        path.write_text(source, encoding="utf-8")
        try:
            result = runner(["bash", str(path), mode], capture_output=True,
                            text=True, timeout=600, check=False)
            receipt["returncode"] = result.returncode
            receipt["status"] = "SUCCEEDED" if result.returncode == 0 else classify_failure(result.stderr, result.returncode)
            # Only known, nonsecret bootstrap messages are eligible for export.
            prefixes = ("Verified identity: ", "Expected provider: ",
                        "Required API disabled: ", "Repository variable ")
            exact = {"Verified existing OIDC provider.", "Dedicated pool absent.",
                     "OIDC provider absent.", "Scoped impersonation binding present.",
                     "Scoped impersonation binding absent.", "GCI_SETUP_READY=true",
                     "GCI_SETUP_READY=false"}
            receipt["checks"] = [line for line in result.stdout.splitlines()
                                 if line in exact or line.startswith(prefixes)]
            receipt["ready"] = result.returncode == 0 and "GCI_SETUP_READY=true" in receipt["checks"]
        except subprocess.TimeoutExpired:
            receipt["status"] = "TIMED_OUT_STATE_UNKNOWN"
            receipt["next_action"] = "Run read-only audit before any further apply."
        except OSError:
            receipt["status"] = "RUNTIME_COMMAND_UNAVAILABLE"
    receipt["finished_utc"] = datetime.now(timezone.utc).isoformat()
    return receipt


def launch(source, digest):
    """Render controls; displaying or Run all never changes IAM."""
    import ipywidgets as widgets
    from IPython.display import display
    from google.colab import auth, files

    state = {"audited": False, "busy": False, "receipts": []}
    audit = widgets.Button(description="1. Sign in and check", layout=widgets.Layout(width="250px"))
    apply = widgets.Button(description="2. Apply reviewed setup", disabled=True, layout=widgets.Layout(width="250px"))
    verify = widgets.Button(description="3. Verify setup", disabled=True, layout=widgets.Layout(width="250px"))
    download = widgets.Button(description="Download safe receipt", disabled=True, layout=widgets.Layout(width="250px"))
    output = widgets.Output()

    def execute(mode):
        if state["busy"] or (mode != "--audit" and not state["audited"]):
            return
        state["busy"] = True
        state["audited"] = False
        for button in (audit, apply, verify, download):
            button.disabled = True
        with output:
            output.clear_output()
            try:
                missing = [name for name in ("gcloud", "jq", "bash") if not shutil.which(name)]
                if missing:
                    raise SetupPrerequisiteError("Runtime dependencies missing: " + ", ".join(missing))
                # Do not fall back to the old copy-an-auth-code terminal flow.
                if os.environ.get("USE_AUTH_EPHEM") != "1":
                    raise SetupPrerequisiteError("This runtime lacks Colab's dialog sign-in. Stop and report this message.")
                auth.authenticate_user(project_id=PROJECT)
                receipt = run_setup(source, digest, mode)
                state["receipts"].append(receipt)
                state["audited"] = receipt["status"] == "SUCCEEDED"
                print(json.dumps(receipt, indent=2))
                if receipt["ready"]:
                    print("Google setup verified. Return to GCI-001 with this receipt for GitHub preflight. No Gemini call was made.")
                elif receipt["status"] == "SUCCEEDED":
                    print("Read-only check completed. Apply is available for the reviewed Test setup.")
                else:
                    print("Stopped. Download the safe receipt. Do not retry Apply until a fresh audit passes.")
            except SetupPrerequisiteError as error:
                state["receipts"].append({"mode": mode, "status": "RUNTIME_PREREQUISITE_MISSING",
                                          "detail": str(error), "ready": False, "gemini_requests": 0})
                print(str(error))
            except Exception as error:
                # OAuth errors can carry sensitive URLs. Do not print exception values.
                state["receipts"].append({"mode": mode, "status": "AUTH_OR_RUNTIME_BLOCKED",
                                          "error_type": type(error).__name__,
                                          "ready": False, "gemini_requests": 0})
                print("Sign-in or runtime blocked. Setup is not verified. Download the safe receipt and run a fresh audit before applying again.")
            finally:
                state["busy"] = False
                audit.disabled = False
                apply.disabled = not state["audited"]
                verify.disabled = not state["audited"]
                download.disabled = not state["receipts"]

    def export(_):
        path = Path("/content/gci-001-access-receipt.json")
        path.write_text(json.dumps(state["receipts"], indent=2) + "\n", encoding="utf-8")
        files.download(str(path))

    audit.on_click(lambda _: execute("--audit"))
    apply.on_click(lambda _: execute("--apply"))
    verify.on_click(lambda _: execute("--verify"))
    download.on_click(export)
    display(widgets.VBox([audit, apply, verify, download, output]))


BOOTSTRAP_SOURCE = '#!/usr/bin/env bash\n# Audit by default. Only --apply can enable APIs or change IAM.\nset -euo pipefail\nPROJECT_ID=comind-513807\nPROJECT_NUMBER=373198958202\nACCOUNT_UNIQUE_ID=109699106806527626952\nPOOL_ID=comind-gci\nPROVIDER_ID=github-comind\nREPO=jlhogarth/CoMind\nREPO_ID=1028814081\nOWNER_ID=83435038\nBRANCH=gci-001-on-demand-gemini-review\nWORKFLOW=.github/workflows/gci-001-on-demand-gemini.yml\nISSUER=https://token.actions.githubusercontent.com\nCONDITION="assertion.repository_id == \'$REPO_ID\' && assertion.repository_owner_id == \'$OWNER_ID\' && assertion.ref == \'refs/heads/$BRANCH\' && assertion.workflow_ref == \'$REPO/$WORKFLOW@refs/heads/$BRANCH\' && assertion.event_name == \'workflow_dispatch\'"\nMAPPING="google.subject=assertion.sub,attribute.repository_id=assertion.repository_id"\nMAPPING+=",attribute.repository_owner_id=assertion.repository_owner_id,attribute.ref=assertion.ref"\nMAPPING+=",attribute.workflow_ref=assertion.workflow_ref,attribute.event_name=assertion.event_name"\nREQUIRED_APIS=(iam.googleapis.com iamcredentials.googleapis.com sts.googleapis.com drive.googleapis.com docs.googleapis.com)\n\nMODE="${1:---audit}"\nif [[ $# -gt 1 || ( "$MODE" != --audit && "$MODE" != --apply && "$MODE" != --verify ) ]]; then\n  echo "Usage: bootstrap-google-oidc.sh [--audit|--apply|--verify]" >&2\n  exit 2\nfi\nfor dependency in gcloud jq; do\n  command -v "$dependency" >/dev/null || { echo "Missing dependency: $dependency" >&2; exit 4; }\ndone\nACCOUNTS="$(gcloud iam service-accounts list --project="$PROJECT_ID" --format=json)"\nEMAIL="$(jq -er --arg uid "$ACCOUNT_UNIQUE_ID" \'[.[] | select(.uniqueId == $uid and .disabled != true)] | if length == 1 then .[0].email else error("Expected exactly one enabled service account for the recorded numeric ID") end\' <<<"$ACCOUNTS")"\nNUMBER="$(gcloud projects describe "$PROJECT_ID" --format=\'value(projectNumber)\')"\n[[ "$NUMBER" == "$PROJECT_NUMBER" ]] || { echo "Project number mismatch. Stop for project review." >&2; exit 3; }\nPOOL="projects/$NUMBER/locations/global/workloadIdentityPools/$POOL_ID"\nPROVIDER="$POOL/providers/$PROVIDER_ID"\nMEMBER="principalSet://iam.googleapis.com/$POOL/attribute.repository_id/$REPO_ID"\nprintf \'Verified identity: %s\\nExpected provider: %s\\n\' "$EMAIL" "$PROVIDER"\n\nENABLED="$(gcloud services list --enabled --project="$PROJECT_ID" --format=\'value(config.name)\')"\nMISSING=()\nfor api in "${REQUIRED_APIS[@]}"; do\n  if ! grep -Fxq "$api" <<<"$ENABLED"; then MISSING+=("$api"); fi\ndone\nif (( ${#MISSING[@]} )); then printf \'Required API disabled: %s\\n\' "${MISSING[@]}"; fi\n\nPOOLS="$(gcloud iam workload-identity-pools list --location=global --project="$PROJECT_ID" --format=json)"\nPOOL_DATA="$(jq -c --arg pool "$POOL" \'.[] | select(.name == $pool)\' <<<"$POOLS")"\nPROVIDER_DATA=\nif [[ -n "$POOL_DATA" ]]; then\n  jq -e \'.state == "ACTIVE" and .disabled != true\' <<<"$POOL_DATA" >/dev/null\n  PROVIDERS="$(gcloud iam workload-identity-pools providers list --location=global --workload-identity-pool="$POOL_ID" --project="$PROJECT_ID" --format=json)"\n  if ! jq -e --arg provider "$PROVIDER" \'all(.[]; .name == $provider)\' <<<"$PROVIDERS" >/dev/null; then\n    echo "Unexpected provider in the dedicated pool. Stop for IAM review." >&2\n    exit 3\n  fi\n  PROVIDER_DATA="$(jq -c --arg provider "$PROVIDER" \'.[] | select(.name == $provider)\' <<<"$PROVIDERS")"\nfi\nif [[ -n "$PROVIDER_DATA" ]]; then\n  if ! jq -e --arg cond "$CONDITION" --arg issuer "$ISSUER" --arg mapping "$MAPPING" \'\n    .attributeCondition == $cond and .oidc.issuerUri == $issuer and\n    ((.oidc.allowedAudiences // []) | length == 0) and\n    .attributeMapping == ($mapping | split(",") | map(split("=") | {key: .[0], value: .[1]}) | from_entries) and\n    .state == "ACTIVE" and .disabled != true\n  \' <<<"$PROVIDER_DATA" >/dev/null; then\n    echo "Existing provider differs from the approved trust rules. Stop for IAM review." >&2\n    exit 3\n  fi\n  echo "Verified existing OIDC provider."\nfi\nPROJECT_POLICY="$(gcloud projects get-iam-policy "$PROJECT_ID" --format=json)"\nif ! jq -e --arg identity "serviceAccount:$EMAIL" \'\n  all(.bindings[]?; (.members | index($identity)) == null)\n\' <<<"$PROJECT_POLICY" >/dev/null; then\n  echo "The test service account has a direct project-level role. Stop for IAM review." >&2\n  exit 3\nfi\nUSER_KEYS="$(gcloud iam service-accounts keys list --iam-account="$EMAIL" --managed-by=user --project="$PROJECT_ID" --format=json)"\nif ! jq -e \'length == 0\' <<<"$USER_KEYS" >/dev/null; then\n  echo "The test service account has a user-managed key. Stop for IAM review." >&2\n  exit 3\nfi\nPOLICY="$(gcloud iam service-accounts get-iam-policy "$EMAIL" --project="$PROJECT_ID" --format=json)"\nif ! jq -e --arg member "$MEMBER" \'\n  all(.bindings[]? | select(.role == "roles/iam.workloadIdentityUser" or .role == "roles/iam.serviceAccountTokenCreator");\n    .role == "roles/iam.workloadIdentityUser" and .condition == null and all(.members[]; . == $member))\n\' <<<"$POLICY" >/dev/null; then\n  echo "Unexpected impersonation grant on the service account. Stop for IAM review." >&2\n  exit 3\nfi\nbinding_present() {\n  jq -e --arg member "$MEMBER" \'\n    any(.bindings[]?; .role == "roles/iam.workloadIdentityUser" and .condition == null and (.members | index($member) != null))\n  \' <<<"$1" >/dev/null\n}\n\nif [[ "$MODE" == --audit || "$MODE" == --verify ]]; then\n  [[ -n "$POOL_DATA" ]] || echo "Dedicated pool absent."\n  [[ -n "$PROVIDER_DATA" ]] || echo "OIDC provider absent."\n  if binding_present "$POLICY"; then echo "Scoped impersonation binding present."; else echo "Scoped impersonation binding absent."; fi\n  if (( ${#MISSING[@]} == 0 )) && [[ -n "$POOL_DATA" && -n "$PROVIDER_DATA" ]] && binding_present "$POLICY"; then\n    echo "GCI_SETUP_READY=true"\n    printf \'Repository variable GCI_WIF_PROVIDER=%s\\nRepository variable GCI_SERVICE_ACCOUNT_EMAIL=%s\\n\' "$PROVIDER" "$EMAIL"\n  else\n    echo "GCI_SETUP_READY=false"\n    [[ "$MODE" != --verify ]] || exit 5\n  fi\n  exit 0\nfi\n\nif (( ${#MISSING[@]} )); then gcloud services enable "${MISSING[@]}" --project="$PROJECT_ID"; fi\nif [[ -z "$POOL_DATA" ]]; then\n  gcloud iam workload-identity-pools create "$POOL_ID" --location=global --project="$PROJECT_ID" --display-name="CoMind GCI GitHub identity"\nfi\nif [[ -z "$PROVIDER_DATA" ]]; then\n  gcloud iam workload-identity-pools providers create-oidc "$PROVIDER_ID" \\\n    --location=global --workload-identity-pool="$POOL_ID" --project="$PROJECT_ID" \\\n    --display-name="CoMind GCI test workflow only" --issuer-uri="$ISSUER" \\\n    --attribute-mapping="$MAPPING" --attribute-condition="$CONDITION"\nfi\nif ! binding_present "$POLICY"; then\n  gcloud iam service-accounts add-iam-policy-binding "$EMAIL" --project="$PROJECT_ID" \\\n    --role=roles/iam.workloadIdentityUser --member="$MEMBER" --condition=None >/dev/null\nfi\n# A successful mutation response alone is insufficient. Re-read all trust state.\nbash "$0" --verify\n\n'
BOOTSTRAP_SHA256 = '3fa61eaa579aa49e709c3c4a6e2d752c4579d31af821a3d712d3cc9141a28468'
launch(BOOTSTRAP_SOURCE, BOOTSTRAP_SHA256)
